# Create Natural Science Foundation of Guangdong Province Awards

Creates awards from the Natural Science Foundation of Guangdong Province's
per-project 拟立项项目 rosters, published by the 广东省基础与应用基础研究基金委员会
on gdstc.gd.gov.cn.

**Prerequisites:** run `scripts/local/guangdong_nsf_to_s3.py` first (thin runner
over the shared `scripts/local/cn_provincial` framework, with the extras.py
Wayback recovery add-on). It uploads
`s3://openalex-ingest/awards/guangdong_nsf/guangdong_nsf_projects.parquet`.

**Data source:** gdstc.gd.gov.cn/zwgk_n/tzgg (通知公告). The committee removes each
roster PDF from the live server after its 公示期; the harvester recovers archived
copies from the Internet Archive. Roster columns: 序号|主管部门|项目名称|申报单位|
负责人|拟立项金额|拨付金额|项目类型.

**Amounts:** PUBLISHED in 万元 (×10,000 CNY). This notebook multiplies
`amount_raw` by 10,000 and sets `currency='CNY'`. §6.7 amount check APPLIES.

**PI names:** Chinese, family-first -> full name in `family_name`, `given_name`
NULL (NSFC precedent).

**Funder details (Path A, F4320*):**
- funder_id: `4320321921`
- display_name: Natural Science Foundation of Guangdong Province / country: CN

**Priority:** `445` (direct funder ingest; higher wins under DESC dedup).


## Step 1: Create Staging Table from S3


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.guangdong_nsf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/guangdong_nsf/guangdong_nsf_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.guangdong_nsf_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.guangdong_nsf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.guangdong_nsf_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320321921 is a Crossref-registered F4320* funder, so it MUST resolve to
exactly 1 row in `openalex.common.funder`. If 0 rows, STOP and flag.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320321921;

## Step 2: Create Natural Science Foundation of Guangdong Province Awards Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.guangdong_nsf_awards
USING delta
AS
WITH
-- Path A: Natural Science Foundation of Guangdong Province is F4320* (Crossref-registered) -> resolve from the dim.
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321921
),

awards_transformed AS (
    SELECT
        -- Key the id hash on funder_award_id when present, else a synthetic
        -- (title + institution) key. Stable across re-ingests.
        abs(xxhash64(CONCAT(
            f.funder_id, ':',
            COALESCE(
                NULLIF(LOWER(TRIM(g.funder_award_id)), ''),
                CONCAT(LOWER(TRIM(g.display_name)), '|', LOWER(TRIM(COALESCE(g.institution, ''))))
            )
        ))) % 9000000000 as id,

        g.display_name as display_name,
        CAST(NULL AS STRING) as description,

        f.funder_id,
        NULLIF(TRIM(g.funder_award_id), '') as funder_award_id,

        -- 万元 (×10,000 CNY) -> whole CNY units.
        CASE WHEN TRY_CAST(g.amount_raw AS DOUBLE) IS NOT NULL
             THEN TRY_CAST(g.amount_raw AS DOUBLE) * 10000
             ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount_raw AS DOUBLE) IS NOT NULL
             THEN 'CNY' ELSE CAST(NULL AS STRING) END as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        -- Funding type from the scheme label (青年/优青/杰青 -> fellowship;
        -- 重点/重大/联合基金 -> research; else default 'grant').
        CASE
            WHEN g.funder_scheme LIKE '%杰出青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%优秀青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%博士%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%启明星%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%扬帆%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%重大%' THEN 'research'
            WHEN g.funder_scheme LIKE '%重点%' THEN 'research'
            WHEN g.funder_scheme LIKE '%联合基金%' THEN 'research'
            ELSE 'grant'
        END as funding_type,

        NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,

        'guangdong_nsf' as provenance,

        -- Dates: rosters publish the approval year -> start = yyyy-01-01, no end.
        CASE WHEN TRY_CAST(g.start_year AS INT) IS NOT NULL
             THEN TRY_TO_DATE(CONCAT(g.start_year, '-01-01'), 'yyyy-MM-dd')
             ELSE NULL END as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.start_year AS INT) as start_year,
        CAST(NULL AS INT) as end_year,

        -- Lead investigator: Chinese PI full name in family_name, given NULL (NSFC precedent).
        CASE
            WHEN (g.lead_family_name IS NOT NULL AND TRIM(g.lead_family_name) != '')
              OR (g.institution IS NOT NULL AND TRIM(g.institution) != '') THEN
                struct(
                    CAST(NULL AS STRING) as given_name,
                    NULLIF(TRIM(g.lead_family_name), '') as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        NULLIF(TRIM(g.institution), '') as name,
                        'China' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.landing_page_url as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G',
               abs(xxhash64(CONCAT(
            f.funder_id, ':',
            COALESCE(
                NULLIF(LOWER(TRIM(g.funder_award_id)), ''),
                CONCAT(LOWER(TRIM(g.display_name)), '|', LOWER(TRIM(COALESCE(g.institution, ''))))
            )
        ))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM openalex.awards.guangdong_nsf_raw g
    CROSS JOIN src_funder f
    WHERE g.display_name IS NOT NULL
      AND TRIM(g.display_name) != ''
)

SELECT * FROM awards_transformed;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'guangdong_nsf' AND priority = 445;

-- Insert into openalex_awards_raw with priority.
-- Priority 445: direct-from-funder ingest. Higher wins under the
-- 2026-06-20 DESC dedup (oxjob #500), so it outranks acknowledgement shells
-- (priority 0) and grant-DOI stubs (priority 1).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    445 as priority
FROM openalex.awards.guangdong_nsf_awards;

## Verification Queries


In [ ]:
%sql
SELECT COUNT(*) as total_guangdong_nsf_awards FROM openalex.awards.guangdong_nsf_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency,
       start_year, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.guangdong_nsf_awards LIMIT 20;

In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt FROM openalex.awards.guangdong_nsf_awards
GROUP BY funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.guangdong_nsf_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- §6.7 coverage. amount published in 万元 -> ×10,000 CNY.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.guangdong_nsf_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (§6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'guangdong_nsf'
GROUP BY provenance, priority;